# Sweden residents — group sizes and a back-of-the-envelope MDE

How many children would the cohort-exposure design
(the pre-spec `docs/lines/sweden-residents/prespec-cohort-exposure.tex`, drafted on `template-port-delta`) have behind each
barrier, on the opposite side, and in the 600–1,500 m band behind it, and what effect could it
detect? **No outcome is loaded:** sizes come from the open population grid
([`population`](../../docs/data/sweden/population/README.md)), not the register, which only the
coauthors with MONA access can count. The numbers are for planning, not for the record.

- **Population:** GHS-POP R2023A (100 m, modelled, epochs 1975–2025) at the reference year
  ρ_b = τ_b − 3, interpolated between epochs, times each cell's share aged 0–19 in SCB's
  open 1 km grid (2015–2025), divided by 20: children per birth cohort.
- **Groups and stacks:** `noise_pollution.analysis.sweden_residents` (rules in its docstring):
  one stack per dated barrier, protected cells of the barrier's year, clean comparison cells on
  the opposite side (main) or in the 600–1,500 m band (secondary).
- **MDE:** δ^SR on the short window (e = −4…+3) and δ, β (per year of dose), β^utero on the full
  window (e = −4…17), as specified in the pre-spec's §"Effect per year of dose"; 5% two-sided,
  80% power, treated-share weights.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "pyproject.toml").is_file():
            return c
    raise FileNotFoundError(f"repo root not found from {Path.cwd()}")


ROOT = _find_repo_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from noise_pollution.analysis.spec import np, pd

import geopandas as gpd

from noise_pollution.analysis import sweden_residents as SR
from noise_pollution.regions.sweden.sources._layout import METRIC_CRS
from noise_pollution.regions.sweden.sources.barrier_protection.shared import load_barrier_references
from noise_pollution.regions.sweden.sources.grid.shared import processed_grid_cells_path
from noise_pollution.regions.sweden.sources.noise_barriers.shared import BARRIER_KINDS, load_noise_barriers
from noise_pollution.regions.sweden.sources.population import shared as POP

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
ROOT

PosixPath('/Users/felixschulz/Developer/noise-pollution')

## 1. The population inputs

GHS-POP spreads SCB's 2010 parish counts over built-up volume and carries them to other epochs
with growth rates; SCB's 1 km grid is register counts. Where both exist (2015–2025) they should
agree on the corridor's total; GHS's earlier epochs are only as good as its built-up layers.

In [2]:
cells = gpd.read_parquet(processed_grid_cells_path(ROOT))
ghs = pd.read_parquet(POP.grid_population_ghs_path(ROOT))
scb = pd.read_parquet(POP.grid_population_scb_path(ROOT))
assert (ghs["cell_id"].to_numpy() == cells["cell_id"].to_numpy()).all()

epochs = [int(c.split("_")[1]) for c in ghs.columns if c.startswith("pop_")]
check = pd.DataFrame({"GHS-POP": [ghs[f"pop_{e}"].sum() for e in epochs]}, index=pd.Index(epochs, name="year"))
check["SCB (1 km, spread by GHS)"] = scb.groupby("year")["pop"].sum().reindex(epochs)
check["SCB aged 0-19"] = scb.assign(k=scb[list(SR.CHILD_AGE_COLUMNS)].sum(axis=1)).groupby("year")["k"].sum().reindex(epochs)
print(f"{len(cells):,} grid cells (within 1.6 km of a barrier); residents in them:")
check.style.format("{:,.0f}", na_rep="")

832,221 grid cells (within 1.6 km of a barrier); residents in them:


,GHS-POP,"SCB (1 km, spread by GHS)",SCB aged 0-19
year,,,
1975,"3,003,924",,
1980,"3,011,230",,
1985,"3,032,166",,
1990,"3,081,688",,
1995,"3,173,213",,
2000,"3,232,805",,
2005,"3,310,160",,
2010,"3,459,068",,
2015,"3,645,898","3,649,198","805,608"


## 2. Cells and barriers: protected, opposite side, 600–1,500 m band

Every (cell, barrier) pair where the barrier protects the cell, would protect it from across the
road (one-sided barriers only), or holds it in its control band. ~3 minutes.

In [3]:
treatment = pd.read_parquet(ROOT / "data/sweden/grid/assembled/grid_treatment_protected.parquet")
assert (treatment["cell_id"].to_numpy() == cells["cell_id"].to_numpy()).all()
points = cells.geometry.to_crs(METRIC_CRS).to_numpy()

pairs, status, built = {}, {}, {}
for kind in BARRIER_KINDS:
    barriers = load_noise_barriers(kind, ROOT)
    refs = load_barrier_references(kind, ROOT, barriers_gdf=barriers)
    pairs[kind] = SR.barrier_cell_pairs(points, barriers.to_crs(METRIC_CRS).reset_index(drop=True), refs)
    status[kind] = treatment[[f"{kind}_{c}" for c in ("treated", "treat_year", "timing_unknown", "side_unknown")]].set_axis(
        ["treated", "treat_year", "timing_unknown", "side_unknown"], axis=1
    )
    built[kind] = pd.to_numeric(barriers["built_year"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    one_sided = (~refs.table["side_method"].isin(("unknown", "both_sides", "osm_both_sides", "manual_both_sides"))).sum()
    print(f"{kind}: {len(barriers):,} barriers ({one_sided:,} one-sided), pairs:", pairs[kind]["group"].value_counts().to_dict())

road: 2,172 barriers (1,603 one-sided), pairs: {'band': 39787, 'protected': 25253, 'opposite': 23221}


rail: 1,829 barriers (1,414 one-sided), pairs: {'band': 35509, 'protected': 23456, 'opposite': 20235}


## 3. Cell counts per group

Per barrier kind and window: the stacks' cells after the clean-control rules, by distance from
the barrier's road (the pre-spec's bands r). *Cells* counts distinct cells; *with residents*
those with at least one GHS-POP resident at ρ_b; *children / cohort* sums over stacks (a
comparison cell can serve several stacks, as in the stacked regression).

In [4]:
years = range(1990, 2026)
children = SR.children_per_cohort(ghs, scb, years)
people = pd.DataFrame({y: SR.interpolate_ghs(ghs, y) for y in years}, index=ghs.index)


def at_rho(frame, stack):
    col = {y: i for i, y in enumerate(frame.columns)}
    return frame.to_numpy()[stack["cell"].to_numpy(), [col[r] for r in stack["rho"]]]


rows, stacks = [], {}
for kind in BARRIER_KINDS:
    for window in SR.WINDOWS:
        s = SR.stack_cells(pairs[kind], status[kind], built[kind], window)
        s["children"] = at_rho(children, s)
        s["people"] = at_rho(people, s)
        s["band_r"] = pd.cut(s["lateral_m"], SR.DISTANCE_BANDS, labels=["0-100", "100-250", "250-600", "600-1500"], include_lowest=True)
        stacks[kind, window] = s
        for (group, band), g in s.groupby(["group", "band_r"], observed=True):
            rows.append({
                "kind": kind, "window": f"{window} {SR.WINDOWS[window][0]}-{SR.WINDOWS[window][1]}", "group": group, "r (m)": band,
                "cells": g["cell"].nunique(), "with residents": g.loc[g["people"] >= 1, "cell"].nunique(),
                "people at rho": g["people"].sum(), "children / cohort": g["children"].sum(),
            })
counts = pd.DataFrame(rows)
order = {"protected": 0, "opposite": 1, "band": 2}
counts = counts.sort_values(["kind", "window", "group", "r (m)"], key=lambda c: c.map(order) if c.name == "group" else c)
counts.style.format({"people at rho": "{:,.0f}", "children / cohort": "{:,.0f}", "cells": "{:,}", "with residents": "{:,}"}).hide(axis="index")

kind,window,group,r (m),cells,with residents,people at rho,children / cohort
rail,full 2000-2005,protected,0-100,503,327,"5,693",62
rail,full 2000-2005,protected,100-250,752,468,"10,497",117
rail,full 2000-2005,protected,250-600,"1,811",939,"20,778",231
rail,full 2000-2005,opposite,0-100,271,157,"2,484",26
rail,full 2000-2005,opposite,100-250,411,198,"4,709",51
rail,full 2000-2005,opposite,250-600,923,404,"10,340",110
rail,full 2000-2005,band,600-1500,"5,004","1,618","34,659",381
rail,short 2000-2019,protected,0-100,"1,322",774,"14,291",161
rail,short 2000-2019,protected,100-250,"1,979","1,097","28,656",327
rail,short 2000-2019,protected,250-600,"4,645","2,211","60,182",698


In [5]:
totals = counts.groupby(["kind", "window", "group"], sort=False)[["cells", "with residents", "people at rho", "children / cohort"]].sum()
totals.style.format("{:,.0f}")

## 4. Stacks: how many barriers have children on both sides?

Children per cohort per stack, at ρ_b. A stack enters the regression only with children on both
sides; below one expected child per cohort a side is empty in most cohorts.

In [6]:
sizes = {key: SR.stack_sizes(s, children) for key, s in stacks.items()}
rows = []
for (kind, window), sz in sizes.items():
    p = sz["protected"]
    for comp in ("opposite", "band", "both"):
        c = sz[["opposite", "band"]].sum(axis=1) if comp == "both" else sz[comp]
        use = (p >= SR.MIN_PER_COHORT) & (c >= SR.MIN_PER_COHORT)
        rows.append({
            "kind": kind, "window": window, "comparison": comp, "dated barriers": len(sz),
            "with >=1 protected child / cohort": int((p >= SR.MIN_PER_COHORT).sum()),
            "usable stacks": int(use.sum()),
            "protected children / cohort in them": p[use].sum(),
            "share of protected children kept": p[use].sum() / p.sum(),
            "top-10 stacks' share": p[use].nlargest(10).sum() / p[use].sum() if use.any() else np.nan,
        })
stack_table = pd.DataFrame(rows)
stack_table.style.format({"protected children / cohort in them": "{:,.0f}", "share of protected children kept": "{:.0%}", "top-10 stacks' share": "{:.0%}"}).hide(axis="index")

kind,window,comparison,dated barriers,with >=1 protected child / cohort,usable stacks,protected children / cohort in them,share of protected children kept,top-10 stacks' share
road,short,opposite,477,96,27,140,27%,70%
road,short,band,477,96,56,349,67%,43%
road,short,both,477,96,74,409,78%,39%
road,full,opposite,50,5,4,11,72%,100%
road,full,band,50,5,0,0,0%,nan%
road,full,both,50,5,4,11,72%,100%
rail,short,opposite,717,244,95,532,45%,33%
rail,short,band,717,244,146,779,66%,24%
rail,short,both,717,244,190,926,78%,20%
rail,full,opposite,258,101,35,171,42%,59%


## 5. Minimum detectable effects

Per stack, the estimate is a contrast of protected and comparison children across event times,
with the pre-spec's regressors on it; its variance is σ² · [(X′X)⁻¹]ₖₖ · Σ_side (ICC + (1 − ICC)/n),
n = children per cohort on that side. Stacks are pooled with treated-share weights.

- **σ = 1**: the outcome in SD units, no covariates. Covariates (parental education, income,
  immigrant background) that explain a share R² of the variance shrink every MDE by √(1 − R²).
- **ICC**: correlation of children in the same barrier × side × cohort cell beyond the fixed
  effects (shared peers, school and neighbourhood shocks). 0 is the floor; 0.02 a guess.
- **Percentile points**: the pre-spec's outcome is a percentile rank (SD 28.9).
- The inverse-variance column is the MDE under weights that favour large stacks; the
  treated-share weights of the pre-spec are never more precise.

In [7]:
targets = [("short", "delta_SR"), ("full", "delta"), ("full", "beta_per_year"), ("full", "beta_utero")]
print("variance factors [(X'X)^-1]_kk:", {f"{w}/{c}": round(SR.variance_factor(SR.DESIGNS[w], c), 3) for w, c in targets})
# road + rail pooled in one regression: not in the pre-spec (rail against road is one of its
# tests), shown for what pooling would buy
for window in SR.WINDOWS:
    sizes["road+rail", window] = pd.concat([sizes["road", window], sizes["rail", window]], ignore_index=True)
rows = []
for kind in (*BARRIER_KINDS, "road+rail"):
    for window, coef in targets:
        for comp in ("opposite", "band", "both"):
            for icc in (0.0, 0.02):
                r = SR.mde(sizes[kind, window], comp, SR.DESIGNS[window], coef, icc=icc)
                rows.append({"kind": kind, "window": window, "coefficient": coef, "comparison": comp, "ICC": icc, **r})
mde = pd.DataFrame(rows)
mde.style.format({
    "protected_per_cohort": "{:,.0f}", "comparison_per_cohort": "{:,.0f}", "protected_lost_share": "{:.0%}",
    "mde_sd": "{:.3f}", "mde_sd_inverse_variance": "{:.3f}", "mde_percentile_points": "{:.1f}",
}, na_rep="").hide(axis="index")

variance factors [(X'X)^-1]_kk: {'short/delta_SR': 1.167, 'full/delta': 1.108, 'full/beta_per_year': 0.003, 'full/beta_utero': 1.228}


kind,window,coefficient,comparison,ICC,stacks,protected_per_cohort,comparison_per_cohort,protected_lost_share,mde_sd,mde_sd_inverse_variance,mde_percentile_points
road,short,delta_SR,opposite,0.000000,27,140,148,73%,0.460,0.414,13.3
road,short,delta_SR,opposite,0.020000,27,140,148,73%,0.482,0.434,13.9
road,short,delta_SR,band,0.000000,56,349,609,33%,0.282,0.229,8.1
road,short,delta_SR,band,0.020000,56,349,609,33%,0.297,0.251,8.6
road,short,delta_SR,both,0.000000,74,409,770,22%,0.222,0.204,6.4
road,short,delta_SR,both,0.020000,74,409,770,22%,0.239,0.223,6.9
road,full,delta,opposite,0.000000,4,11,13,28%,1.228,1.217,35.4
road,full,delta,opposite,0.020000,4,11,13,28%,1.256,1.244,36.2
road,full,delta,band,0.000000,0,0,0,100%,,,
road,full,delta,band,0.020000,0,0,0,100%,,,


## 6. Summary (run of 2026-11-19)

Planning numbers from open data; no outcome loaded. MDEs at 5% (two-sided) and 80% power,
outcome in SD units without covariates; ICC 0 unless said.

1. **Inputs.** In the grid's cells GHS-POP and SCB agree within 1% in 2015, 2020 and 2025
   (3.65/3.65, 3.85/3.88, 3.98/4.01 million). Before 2015 there is only GHS-POP, extrapolated
   from SCB's 2010 parish counts.
2. **Children per birth cohort** at ρ_b, summed over stacks (protected / opposite side / 600–1,500 m band):

   | | short window, barriers 2000–2019 | full window, barriers 2000–2005 |
   |---|---|---|
   | road | 523 / 281 / 669 | 16 / 16 / 7 |
   | rail | 1,186 / 547 / 1,133 | 410 / 187 / 381 |

3. **The opposite side is thin.** Only 27 road and 95 rail stacks have at least one child per
   cohort on both sides; they hold 27% (road) and 45% (rail) of the protected children. The band
   keeps 67% / 66%, the two together 78%.
4. **Short-run step δ^SR** (short window), opposite / band / both:
   - road 0.46 / 0.28 / 0.22 SD;
   - rail 0.25 / 0.18 / 0.15 SD;
   - road and rail pooled (not in the pre-spec) 0.22 / 0.15 / 0.13 SD, about 4 percentile points
     for "both".

   An ICC of 0.02 raises these by about 5%; covariates explaining 25% of the variance lower
   them by about 13%.
5. **Full window.** Road: 4 usable stacks with 11 protected children per cohort, so it cannot be
   estimated. Rail, opposite / band / both:
   - step δ 0.38 / 0.33 / 0.26 SD;
   - β per year of dose 0.019 / 0.017 / 0.013 SD (about 0.2–0.3 SD over 16 years);
   - β^utero 0.40 / 0.35 / 0.27 SD.
6. **What it means.** As specified (opposite side main, road and rail separate), the design
   detects only large effects: about 0.25 SD for rail and 0.46 SD for road in the short run.
   Only the band as an equal comparison and pooling road and rail bring this to about 0.13 SD.
   Both would be changes to the pre-spec, to be decided before any estimate.

**Assumptions behind these numbers:**
- Within each km, children are placed by GHS-POP's built-up layers, not by real addresses.
- Children assigned through the mother's cell at ρ_b are approximated by the children living
  there at ρ_b.
- A stack needs at least one expected child per cohort on each side.
- Comparison cells shared by several stacks are treated as independent.

The register counts on MONA settle all of this.